# RFBBP case study: three-biomarker final analysis

## 1. Context and methods

### 1.1 Key assumptions

In [ ]:
import os
from pathlib import Path

for name in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(name, "1")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib_rfbbp_final")

import sys
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from matplotlib.patches import Patch
from IPython.display import display

cwd = Path.cwd().resolve()

if cwd.name == "analysis_final":
    # Running from case_multivar/analysis_final
    ROOT = cwd.parent
elif cwd.name == "case_multivar":
    # Running from case_multivar
    ROOT = cwd
elif (cwd / "case_multivar").is_dir():
    # Running from the repository root
    ROOT = cwd / "case_multivar"
else:
    raise FileNotFoundError(
        "Could not locate the case_multivar directory. Run this notebook from "
        "the repository root, case_multivar, or case_multivar/analysis_final."
    )

SIMPLE = ROOT / "analysis_final"
sys.path.insert(0, str(SIMPLE))
from rfbbp_helpers import (
    read_and_merge_data, add_sofa_adjusted_variables, build_variable_dataset,
    center_description, fit_stratified_cox, fit_within_center_z_cox,
    fit_two_stage_meta, covariates_for, _cox,
)

DATA_PATH = ROOT / "data" / "combined_dataset_raw.csv"
FINAL = ROOT / "analysis_final"
RFBBP_RESULT = FINAL / "RFBBP_result"
TABLES = FINAL / "tables"
FIGURES = FINAL / "figures"
DATASETS = FINAL / "analysis_datasets"
for folder in [TABLES, FIGURES, DATASETS]:
    folder.mkdir(parents=True, exist_ok=True)

BIOMARKERS = {
    "Hemoglobin_hour_24": {"label": "Hemoglobin", "unit": "g/dL", "scale": "raw"},
    "Lactate_hour_24": {"label": "Lactate", "unit": "mmol/L", "scale": "log"},
    "Creatinine_hour_24": {"label": "Creatinine", "unit": "mg/dL", "scale": "log"},
}
COLORS = {"eicu": "#2a9d55", "mimic_iv": "#e76f00", "rfbbp": "#2b7bba", "naive": "#e07a34"}
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#dfe5ec", "grid.linewidth": 0.8,
                     "font.size": 10, "axes.titleweight": "bold"})

## 2. Analysis datasets

### 2.1 Build the three SOFA-adjusted complete-case cohorts

In [ ]:
raw_data, input_info = read_and_merge_data(DATA_PATH)
raw_data = add_sofa_adjusted_variables(raw_data)

cohorts = {}
cohort_rows = []
for variable, spec in BIOMARKERS.items():
    cohort, info = build_variable_dataset(raw_data, variable, adjustment="sofa_adjusted")
    cohorts[variable] = cohort
    cohort.to_csv(DATASETS / f"{variable}_sofa_adjusted.csv", index=False)
    cohort_rows.append({"indicator": spec["label"], "n": len(cohort),
                        "events": int(cohort.event.sum()), "centres": cohort.center_id.nunique(),
                        "analysis_scale": info["transform"], "covariates": ", ".join(info["covariates"])})

cohort_summary = pd.DataFrame(cohort_rows)
cohort_summary.to_csv(TABLES / "cohort_summary.csv", index=False)
display(cohort_summary)

### 2.2 Cohort-level descriptive statistics

In [ ]:
def format_n_pct(n, denominator):
    n, denominator = int(n), int(denominator)
    return f"{n} ({100*n/denominator:.1f})" if denominator else "NA"

def format_n_over_n_pct(n, denominator):
    n, denominator = int(n), int(denominator)
    return f"{n}/{denominator} ({100*n/denominator:.1f})" if denominator else "NA"

def format_median_iqr(values, digits=1):
    values = pd.to_numeric(pd.Series(values), errors="coerce").dropna()
    if values.empty:
        return "NA"
    q1, median, q3 = values.quantile([.25, .50, .75])
    return f"{median:.{digits}f} ({q1:.{digits}f}-{q3:.{digits}f})"

def valid_measurement_denominator(variable):
    sofa_adjustment = "SOFA_no_renal" if variable == "Creatinine_hour_24" else "SOFA_score_hour_24"
    required = ["database", "center_id", "time", "event", "age", "gender_male", sofa_adjustment]
    denominator = raw_data.replace([np.inf, -np.inf], np.nan).dropna(subset=required).copy()
    denominator = denominator.loc[denominator.time.gt(0) & denominator.event.isin([0, 1])]
    return denominator

def valid_measurement_mask(values, variable):
    values = pd.to_numeric(values, errors="coerce")
    valid = values.notna()
    if variable in {"Lactate_hour_24", "Creatinine_hour_24"}:
        valid &= values.gt(0)
    return valid

def cohort_descriptive_table(variable, cohort):
    spec = BIOMARKERS[variable]
    denominator = valid_measurement_denominator(variable)
    d = cohort.copy()
    # The creatinine analysis dataset contains SOFA excluding the renal component;
    # recover full SOFA for the requested descriptive row without changing the model.
    if "SOFA_score_hour_24" not in d.columns:
        sofa_lookup = raw_data[["database", "stay_id", "SOFA_score_hour_24"]].drop_duplicates(["database", "stay_id"])
        d = d.merge(sofa_lookup, on=["database", "stay_id"], how="left", validate="one_to_one")

    groups = {
        "MIMIC-IV": (d.loc[d.database.eq("mimic_iv")], denominator.loc[denominator.database.eq("mimic_iv")]),
        "eICU-CRD": (d.loc[d.database.eq("eicu")], denominator.loc[denominator.database.eq("eicu")]),
        "Overall": (d, denominator),
    }
    marker_row = f"{spec['label']}, median (IQR), {spec['unit']}"
    row_order = [
        "Valid measurement rate, n/N (%)", "Patients, n", "Centers, n",
        "28-day mortality, n (%)", "Age, median (IQR), years", "Male sex, n (%)",
        "SOFA score within first 24 h, median (IQR)", marker_row,
    ]
    table = pd.DataFrame({"Characteristics": row_order})
    for group_name, (sub, denom) in groups.items():
        valid_n = int(valid_measurement_mask(denom[variable], variable).sum())
        summary = {
            "Valid measurement rate, n/N (%)": format_n_over_n_pct(valid_n, len(denom)),
            "Patients, n": str(len(sub)),
            "Centers, n": str(sub.center_id.nunique()),
            "28-day mortality, n (%)": format_n_pct(sub.event.sum(), len(sub)),
            "Age, median (IQR), years": format_median_iqr(sub.age, 1),
            "Male sex, n (%)": format_n_pct(sub.gender_male.sum(), len(sub)),
            "SOFA score within first 24 h, median (IQR)": format_median_iqr(sub.SOFA_score_hour_24, 1),
            marker_row: format_median_iqr(sub.exposure_raw, 1),
        }
        table[group_name] = table.Characteristics.map(summary)
    return table

descriptive_tables = {}
descriptive_long = []
for variable, cohort in cohorts.items():
    label = BIOMARKERS[variable]["label"]
    table = cohort_descriptive_table(variable, cohort)
    descriptive_tables[variable] = table
    table.to_csv(TABLES / f"descriptive_statistics_{variable}_sofa_adjusted.csv", index=False)
    display(label)
    display(table.set_index("Characteristics"))
    descriptive_long.append(table.assign(variable=variable, cohort=label))

descriptive_summary = pd.concat(descriptive_long, ignore_index=True)
descriptive_summary.to_csv(TABLES / "three_biomarker_descriptive_summary_long.csv", index=False)

## 3. Centre description and screening


In [ ]:
centre_tables = []
for variable, cohort in cohorts.items():
    table = center_description(cohort, variable, n_min=1, event_min=0, nonevent_min=0)
    centre_tables.append(table)

centre_long = pd.concat(centre_tables, ignore_index=True)
counts = centre_long.pivot(index=["database", "center_id"], columns="variable",
                           values=["n", "event_n", "nonevent_n"])
counts.columns = [f"{BIOMARKERS[var]['label'].lower()}_{metric}" for metric, var in counts.columns]
initial_counts = (raw_data.groupby(["database", "center_id"], dropna=False)
                  .agg(initial_total=("event", "size"), initial_events=("event", "sum"))
                  .reset_index())
initial_counts["initial_events"] = initial_counts.initial_events.astype(int)
initial_counts["initial_nonevents"] = initial_counts.initial_total - initial_counts.initial_events
centre_counts_wide = initial_counts.merge(counts.reset_index(), on=["database", "center_id"], how="left")
count_columns = [column for column in centre_counts_wide.columns if column not in {"database", "center_id"}]
centre_counts_wide[count_columns] = centre_counts_wide[count_columns].fillna(0).astype(int)
centre_counts_wide = (centre_counts_wide
                      .sort_values(["initial_total", "database", "center_id"],
                                   ascending=[False, True, True])
                      .reset_index(drop=True))
centre_counts_wide.to_csv(TABLES / "centre_sample_event_nonevent_three_biomarkers_wide.csv", index=False)

centre_distribution = (centre_long.groupby("variable")
    .agg(centres=("center_id", "nunique"), n_median=("n", "median"), n_q1=("n", lambda x: x.quantile(.25)),
         n_q3=("n", lambda x: x.quantile(.75)), event_rate_median=("event_rate", "median"),
         exposure_median_across_centres=("exposure_median", "median"))
    .reset_index())
centre_distribution["indicator"] = centre_distribution.variable.map(lambda x: BIOMARKERS[x]["label"])
centre_distribution.to_csv(TABLES / "centre_descriptive_summary_three_biomarkers.csv", index=False)
display(centre_distribution.drop(columns="variable"))
display(centre_counts_wide.head(12))

## 4. Traditional Cox models

In [ ]:
cox_rows = []
for variable, cohort in cohorts.items():
    covariates = covariates_for(variable, "sofa_adjusted")
    for method, fitter, term in [
        ("Naive", fit_stratified_cox, "exposure_analysis"),
        ("Naive_std", fit_within_center_z_cox, "exposure_center_z"),
    ]:
        terms, model_info = fitter(cohort, covariates)
        estimate = terms.loc[terms.term.eq(term)].iloc[0]
        cox_rows.append({"variable": variable, "indicator": BIOMARKERS[variable]["label"],
                         "method": method, "log_hr": estimate.coef, "se": estimate.se,
                         "hr": estimate.hr, "ci_low": estimate.ci_low, "ci_high": estimate.ci_high,
                         "p": estimate.p, "n": model_info["n"], "events": model_info["events"]})

cox_results = pd.DataFrame(cox_rows)
cox_results.to_csv(TABLES / "traditional_stratified_cox_three_biomarkers.csv", index=False)
display(cox_results)

## 5. Two-stage random-effects meta-analysis

In [ ]:
meta_rows = []
two_stage_center_results_by_variable = {}
for variable, cohort in cohorts.items():
    centre_effects, meta = fit_two_stage_meta(cohort, covariates_for(variable, "sofa_adjusted"))
    centre_effects.insert(0, "variable", variable)
    centre_effects.insert(1, "indicator", BIOMARKERS[variable]["label"])
    two_stage_center_results_by_variable[variable] = centre_effects.copy()
    centre_effects.to_csv(TABLES / f"two_stage_centre_effects_{variable}.csv", index=False)
    random_effect = meta.loc[meta.meta_model.eq("random_effect")].iloc[0].to_dict()
    random_effect.update({"variable": variable, "indicator": BIOMARKERS[variable]["label"]})
    meta_rows.append(random_effect)

meta_results = pd.DataFrame(meta_rows)
meta_results.to_csv(TABLES / "two_stage_meta_three_biomarkers.csv", index=False)
display(meta_results[["indicator", "center_models_n", "hr", "ci_low", "ci_high", "tau2", "I2"]])

# Results

## 6. Cross-centre differences before RFBBP

In [ ]:
source_tables = RFBBP_RESULT

def residualize(y, x):
    design = np.column_stack([np.ones(len(x)), x])
    return y - design @ np.linalg.lstsq(design, y, rcond=None)[0]

def incremental_r2(y, target, adjust):
    reduced_residual = residualize(y, adjust)
    full_design = np.column_stack([adjust, target])
    full_residual = residualize(y, full_design)
    return 1 - np.sum(full_residual**2) / np.sum(reduced_residual**2)

partial_rows = []
permutation_rows = []
rng = np.random.default_rng(20260923)
N_PERMUTATIONS = 999

for variable, cohort in cohorts.items():
    label = BIOMARKERS[variable]["label"]
    sofa_col = "SOFA_no_renal_z" if variable == "Creatinine_hour_24" else "SOFA_score_hour_24_z"
    y = cohort.exposure_analysis.to_numpy(float)
    age = cohort.age_z.to_numpy(float)[:, None]
    sex = cohort.gender_male.to_numpy(float)[:, None]
    sofa = cohort[sofa_col].to_numpy(float)[:, None]
    centre = pd.get_dummies(cohort.center_id, drop_first=True, dtype=float).to_numpy()
    blocks = {"Age": age, "Sex": sex, "SOFA": sofa, "Center": centre}
    for predictor, target in blocks.items():
        other = [value for name, value in blocks.items() if name != predictor]
        value = incremental_r2(y, target, np.column_stack(other))
        partial_rows.append({"variable": variable, "biomarker_label": label,
                             "predictor": predictor, "partial_R2_percent": 100 * value,
                             "n": len(cohort), "centres": cohort.center_id.nunique()})

    # Use the same incremental partial-R² definition for Panel A and Panel B.
    # The cross-product calculation below is algebraically equivalent to
    # incremental_r2(y, centre_dummies, adjust), but avoids constructing a
    # large dummy matrix and refitting a patient-level model 1,000 times.
    adjust = np.column_stack([age, sex, sofa])
    reduced_design = np.column_stack([np.ones(len(y)), adjust])
    reduced_beta = np.linalg.lstsq(reduced_design, y, rcond=None)[0]
    reduced_residual = y - reduced_design @ reduced_beta
    reduced_sse = np.sum(reduced_residual**2)
    reduced_crossprod_inverse = np.linalg.pinv(reduced_design.T @ reduced_design)
    centre_codes, centre_levels = pd.factorize(cohort.center_id, sort=True)

    def center_partial_r2_from_codes(codes):
        n_centres = len(centre_levels)
        counts = np.bincount(codes, minlength=n_centres).astype(float)
        group_design = np.column_stack([
            np.bincount(codes, weights=reduced_design[:, column], minlength=n_centres)
            for column in range(reduced_design.shape[1])
        ])
        group_residual = np.bincount(codes, weights=reduced_residual, minlength=n_centres)
        # Match pd.get_dummies(..., drop_first=True) by omitting centre code 0.
        group_design = group_design[1:]
        score = group_residual[1:]
        information = (np.diag(counts[1:])
                       - group_design @ reduced_crossprod_inverse @ group_design.T)
        coefficient = np.linalg.lstsq(information, score, rcond=None)[0]
        explained_ss = float(score @ coefficient)
        return explained_ss / reduced_sse

    expected_center_r2 = next(row["partial_R2_percent"] for row in partial_rows
                              if row["variable"] == variable and row["predictor"] == "Center") / 100
    for permutation_id in range(N_PERMUTATIONS + 1):
        codes = centre_codes if permutation_id == 0 else rng.permutation(centre_codes)
        centre_r2 = center_partial_r2_from_codes(codes)
        if permutation_id == 0 and not np.isclose(centre_r2, expected_center_r2, rtol=1e-9, atol=1e-12):
            raise RuntimeError(f"Panel A/B center partial R² mismatch for {label}")
        permutation_rows.append({"variable": variable, "biomarker_label": label,
                                 "permutation": permutation_id, "center_R2_percent": 100 * centre_r2})

partial_r2 = pd.DataFrame(partial_rows)
permutation = pd.DataFrame(permutation_rows)
partial_r2.to_csv(TABLES / "pooled_covariate_partial_r2.csv", index=False)
permutation.to_csv(TABLES / "pooled_center_partial_r2_permutation_null.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13.8, 5.8), dpi=300, gridspec_kw={"width_ratios": [1.10, 1], "wspace": .12})
labels = ["Hemoglobin", "Lactate", "Creatinine"]
y_positions = np.arange(3)[::-1]
max_count = 1
histograms = []
for y, variable, label in zip(y_positions, BIOMARKERS, labels):
    null = permutation.loc[(permutation.variable.eq(variable)) & permutation.permutation.ne(0), "center_R2_percent"].to_numpy()
    counts, edges = np.histogram(null, bins=55)
    max_count = max(max_count, counts.max())
    histograms.append((y, variable, label, counts, edges))
for y, variable, label, counts, edges in histograms:
    observed = permutation.loc[(permutation.variable.eq(variable)) & permutation.permutation.eq(0), "center_R2_percent"].iloc[0]
    null = permutation.loc[(permutation.variable.eq(variable)) & permutation.permutation.ne(0), "center_R2_percent"]
    p_value = (1 + (null >= observed).sum()) / (len(null) + 1)
    mids = (edges[:-1] + edges[1:]) / 2
    baseline = y - .26
    scaled = counts / max_count * .46
    axes[0].fill_between(mids, baseline, baseline + scaled, color="#DCEEEB", alpha=.78)
    axes[0].plot(mids, baseline + scaled, color="#A9CFC8", lw=.8)
    axes[0].plot(observed, y, "o", color="#8F2D68", ms=7)
    axes[0].text(observed + .14, y + .14, f"Observed R² = {observed:.1f}%\nPermutation P {'< 0.001' if p_value < .001 else f'= {p_value:.3f}'}",
                 fontsize=8.4, va="bottom")
axes[0].set(yticks=y_positions, yticklabels=labels, xlabel="Centre partial R² (%)",
            title="A. Observed vs chance-expected centre variation", xlim=(0, 5), ylim=(-.55, 2.9))
for tick in axes[0].get_yticklabels():
    tick.set(fontweight="bold", rotation=90, va="center")
axes[0].grid(axis="x"); axes[0].grid(axis="y", visible=False)
axes[0].spines[["left", "top", "right"]].set_visible(False)
axes[0].tick_params(axis="y", length=0)
axes[0].legend(handles=[
    Patch(facecolor="#DCEEEB", edgecolor="#A9CFC8", label="Random-assignment reference"),
    plt.Line2D([0], [0], marker="o", linestyle="none", color="#8F2D68", label="Observed centre partial R²")],
    frameon=False, loc="upper right", ncol=2)

pivot = partial_r2.pivot(index="biomarker_label", columns="predictor", values="partial_R2_percent").reindex(["Hemoglobin", "Lactate", "Creatinine"])
y = np.arange(len(pivot))[::-1]; height = .16
for j, predictor in enumerate(["Age", "Sex", "SOFA", "Center"]):
    values = pivot[predictor].to_numpy()
    bars = axes[1].barh(y + (1.5-j)*height, values, height,
                 color="#9b2f6f" if predictor == "Center" else ["#d5dbe3", "#c4ccd7", "#aeb8c5"][j], label=predictor)
    for bar, value in zip(bars, values):
        axes[1].text(value + .1, bar.get_y()+bar.get_height()/2, f"{value:.1f}%", va="center", fontsize=8)
axes[1].set(yticks=y, yticklabels=[], xlabel="Additional variation explained after adjustment for the other factors (%)",
            title="B. Partial R² of centre and patient factors")
axes[1].legend(frameon=False, ncol=4, loc="upper right")
axes[1].grid(axis="x", color="#D9DEE7", linewidth=.55, alpha=.38)
axes[1].grid(axis="y", color="#D9DEE7", linewidth=.45, alpha=.28)
fig.subplots_adjust(left=.075, right=.99, top=.90, bottom=.14)
fig.savefig(FIGURES / "cross_centre_partial_r2_and_permutation.png", bbox_inches="tight"); plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9.2, 10.2), dpi=300)
effect_axis_specs = {
    "Hemoglobin_hour_24": (0.10, 10.0, [0.1, 0.2, 0.5, 1, 2, 5, 10]),
    "Lactate_hour_24": (0.10, 30.0, [0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 30]),
    "Creatinine_hour_24": (0.10, 20.0, [0.1, 0.2, 0.5, 1, 2, 5, 10, 20]),
}
for ax, (variable, spec) in zip(axes, BIOMARKERS.items()):
    centre_effects = pd.read_csv(TABLES / f"two_stage_centre_effects_{variable}.csv")
    # Use exactly the same estimability definition as the primary two-stage meta:
    # no n/event/nonevent threshold and no exclusion based on CI width.
    centre_effects = centre_effects.loc[
        centre_effects.status.eq("ok")
        & np.isfinite(centre_effects.coef)
        & np.isfinite(centre_effects.se)
        & centre_effects.se.gt(0)
    ].copy()
    centre_effects["hr"] = np.exp(np.clip(centre_effects.coef, -700, 700))
    centre_effects["ci_low"] = np.exp(np.clip(centre_effects.coef - 1.96 * centre_effects.se, -700, 700))
    centre_effects["ci_high"] = np.exp(np.clip(centre_effects.coef + 1.96 * centre_effects.se, -700, 700))
    display_low, display_high, display_ticks = effect_axis_specs[variable]
    centre_effects = centre_effects.sort_values("hr").reset_index(drop=True)
    x = np.arange(1, len(centre_effects) + 1)

    information = 1 / centre_effects.se.pow(2)
    info_low, info_high = information.quantile([0.05, 0.95])
    info_scaled = ((information - info_low) / max(info_high - info_low, np.finfo(float).eps)).clip(0, 1)
    point_sizes = 9 + 20 * np.sqrt(info_scaled)
    point_rgba = np.tile(np.array([42/255, 157/255, 85/255, 1.0]), (len(centre_effects), 1))
    point_rgba[:, 3] = 0.35 + 0.60 * info_scaled

    ax.vlines(x, centre_effects.ci_low.clip(display_low, display_high),
              centre_effects.ci_high.clip(display_low, display_high), color="#9aa0a6", lw=.65, alpha=.55)
    below = centre_effects.hr.lt(display_low)
    above = centre_effects.hr.gt(display_high)
    within = ~(below | above)
    ax.scatter(x[within], centre_effects.loc[within, "hr"], c=point_rgba[within],
               s=point_sizes[within], marker="o", edgecolors="none", zorder=3)
    ax.scatter(x[below], np.full(below.sum(), display_low), c=point_rgba[below],
               s=point_sizes[below] + 5, marker="v", edgecolors="none", zorder=3)
    ax.scatter(x[above], np.full(above.sum(), display_high), c=point_rgba[above],
               s=point_sizes[above] + 5, marker="^", edgecolors="none", zorder=3)
    ax.axhline(1, color="#6b6b6b", ls="--", lw=1)
    ax.set_yscale("log")
    ax.set_ylim(display_low, display_high)
    ax.set_yticks(display_ticks)
    ax.set_yticklabels([f"{value:g}" for value in display_ticks])
    ax.set_ylabel("Centre-specific hazard ratio")
    primary_meta = meta_results.loc[meta_results.variable.eq(variable)].iloc[0]
    tau2, i2 = float(primary_meta.tau2), float(primary_meta.I2)
    ax.set_title(f"{['A', 'B', 'C'][list(BIOMARKERS).index(variable)]}. {spec['label']}", loc="left", fontweight="bold")
    ax.text(.012, .88, f"centres={len(centre_effects)}\n$\\tau^2$={tau2:.3f}; $I^2$={i2:.1f}%",
            transform=ax.transAxes, va="top", fontsize=8.2,
            bbox={"facecolor": "white", "alpha": .88, "edgecolor": "none", "pad": 2.5})
    ax.text(.988, .04, "Out-of-range estimates shown at boundary; intervals truncated",
            transform=ax.transAxes,
            ha="right", va="bottom", fontsize=6.8, color="#666666")
    ax.grid(axis="y"); ax.grid(axis="x", visible=False)
axes[-1].set_xlabel("Centres ordered by effect estimate")
fig.subplots_adjust(left=.095, right=.985, top=.975, bottom=.065, hspace=.34)
fig.savefig(FIGURES / "pooled_center_specific_cox_effects_three_biomarkers.png", bbox_inches="tight"); plt.show()

## 7. Latent-scale associations and attenuation consistency

In [ ]:
# Main analysis plus five prior-sensitivity specifications for each biomarker.
# Long MCMC runs are opt-in; the command is sequential to avoid concurrent PyMC jobs.
import subprocess

RUN_PRIOR_SENSITIVITY_MCMC = False
REPOSITORY_ROOT = ROOT.parent

PRIOR_SENSITIVITY_BASH = r"""
mkdir -p case_multivar/analysis_final/logs/prior_sensitivity
bash case_multivar/run_rfbbp.sh sensitivity \
  > case_multivar/analysis_final/logs/prior_sensitivity/creatinine_sensitivity_ta095.log 2>&1
""".strip()

print(PRIOR_SENSITIVITY_BASH)
if RUN_PRIOR_SENSITIVITY_MCMC:
    subprocess.run(["bash", "-lc", PRIOR_SENSITIVITY_BASH], cwd=REPOSITORY_ROOT, check=True)

PRIOR_RESULT_DIR = RFBBP_RESULT / "prior_sensitivity"
PRIOR_TABLE_DIR = TABLES / "prior_sensitivity"
PRIOR_TABLE_DIR.mkdir(parents=True, exist_ok=True)

main_analysis_variables = {
    "Hemoglobin": "Hemoglobin_hour_24",
    "Lactate": "Lactate_hour_24",
    "Creatinine": "Creatinine_hour_24",
}
prior_scenarios = [
    ("Hemoglobin", "S1: More diffuse calibration priors", "hb_prior_S1_more_diffuse_calibration"),
    ("Hemoglobin", "S2: Stronger hierarchical shrinkage", "hb_prior_NS2_stronger_hier_shrinkage"),
    ("Hemoglobin", "S3: Weaker hierarchical shrinkage", "hb_prior_S3_weaker_hier_shrinkage"),
    ("Hemoglobin", "S4: Alternative measurement-variability prior", "hb_prior_S4_hn_sigmaw1"),
    ("Hemoglobin", "S5: More concentrated exposure-effect prior", "hb_prior_NS4_betax_sigma1"),
    ("Lactate", "S1: More diffuse calibration priors", "lactate_prior_S1_more_diffuse_calibration_rerun"),
    ("Lactate", "S2: Stronger hierarchical shrinkage", "lactate_prior_NS2_stronger_hier_shrinkage"),
    ("Lactate", "S3: Weaker hierarchical shrinkage", "lactate_prior_S3_weaker_hier_shrinkage"),
    ("Lactate", "S4: Alternative measurement-variability prior", "lactate_prior_S4_hn_sigmaw1"),
    ("Lactate", "S5: More concentrated exposure-effect prior", "lactate_prior_NS4_betax_sigma1"),
    ("Creatinine", "S1: More diffuse calibration priors", "creatinine_prior_S1_more_diffuse_calibration"),
    ("Creatinine", "S2: Stronger hierarchical shrinkage", "creatinine_prior_NS2_stronger_hier_shrinkage"),
    ("Creatinine", "S3: Weaker hierarchical shrinkage", "creatinine_prior_S3_weaker_hier_shrinkage"),
    ("Creatinine", "S4: Alternative measurement-variability prior", "creatinine_prior_S4_hn_sigmaw1"),
    ("Creatinine", "S5: More concentrated exposure-effect prior", "creatinine_prior_NS4_betax_sigma1"),
]


def require_one(directory, pattern, label):
    matches = sorted(directory.glob(pattern))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {label} in {directory} matching {pattern!r}; found {matches}"
        )
    return matches[0]


def read_diagnostic_series(path):
    diagnostics = pd.read_csv(path, index_col=0)
    return diagnostics.iloc[:, 0] if diagnostics.shape[1] == 1 else diagnostics.iloc[0]


def posterior_effect_row(biomarker, analysis, summary_path, diagnostics_path=None):
    posterior = pd.read_csv(summary_path, index_col=0)
    missing = [name for name in ["beta_x", "HR_x"] if name not in posterior.index]
    if missing:
        raise KeyError(f"{summary_path} is missing posterior rows: {missing}")
    beta_x = posterior.loc["beta_x"]
    hr_x = posterior.loc["HR_x"]
    row = {
        "Biomarker": biomarker,
        "analysis": analysis,
        "β_x": float(beta_x["estimate"]),
        "HR": float(hr_x["estimate"]),
        "HR_95%CrI": f"{hr_x['CI_1']:.3f}–{hr_x['CI_2']:.3f}",
        "beta_x_r_hat": float(beta_x.get("r_hat", np.nan)),
        "beta_x_ess_bulk": float(beta_x.get("ess_bulk", np.nan)),
        "divergences": np.nan,
    }
    if diagnostics_path is not None:
        diagnostics = read_diagnostic_series(diagnostics_path)
        row.update({
            "beta_x_r_hat": float(diagnostics["diag_beta_x_r_hat"]),
            "beta_x_ess_bulk": float(diagnostics["diag_beta_x_ess_bulk"]),
            "divergences": int(diagnostics["diag_n_divergent"]),
        })
    return row


# Six model specifications per biomarker: main analysis followed by S1-S5.
analysis_rows = []
for biomarker, variable in main_analysis_variables.items():
    main_summary = RFBBP_RESULT / (
        f"{variable}_sofa_adjusted_mimic_iv_eicu_w_raw_rfbbp_hierarchical_"
        "5interval_no_center_filter_mimic_eicu_summary.csv"
    )
    if not main_summary.exists():
        raise FileNotFoundError(f"Missing main-analysis posterior summary: {main_summary}")
    analysis_rows.append(posterior_effect_row(biomarker, "Main analysis", main_summary))

    for sensitivity_biomarker, analysis, tag in prior_scenarios:
        if sensitivity_biomarker != biomarker:
            continue
        summary_path = require_one(PRIOR_RESULT_DIR, f"*_{tag}_summary.csv", f"summary for {tag}")
        diagnostics_path = require_one(
            PRIOR_RESULT_DIR, f"*_{tag}_diagnostics.csv", f"diagnostics for {tag}"
        )
        analysis_rows.append(
            posterior_effect_row(biomarker, analysis, summary_path, diagnostics_path)
        )

main_and_sensitivity_audit = pd.DataFrame(analysis_rows)
if len(main_and_sensitivity_audit) != 18:
    raise RuntimeError(f"Expected 18 model rows (3 biomarkers × 6 analyses); found {len(main_and_sensitivity_audit)}")

sensitivity_only = main_and_sensitivity_audit.loc[
    main_and_sensitivity_audit["analysis"].ne("Main analysis")
]
if sensitivity_only["divergences"].gt(0).any() or sensitivity_only["beta_x_r_hat"].gt(1.01).any():
    raise RuntimeError("At least one selected prior-sensitivity fit failed the convergence gate.")

main_and_sensitivity_table = main_and_sensitivity_audit[
    ["Biomarker", "analysis", "β_x", "HR", "HR_95%CrI"]
].copy()
main_and_sensitivity_table[["β_x", "HR"]] = main_and_sensitivity_table[["β_x", "HR"]].round(3)
effect_csv = PRIOR_TABLE_DIR / "rfbbp_main_and_prior_sensitivity_effect_summary.csv"
effect_xlsx = PRIOR_TABLE_DIR / "rfbbp_main_and_prior_sensitivity_effect_summary.xlsx"
main_and_sensitivity_table.to_csv(effect_csv, index=False)
main_and_sensitivity_table.to_excel(effect_xlsx, index=False)
display(main_and_sensitivity_table)


### 7.1 RFBBP convergence diagnostics for the main and prior-sensitivity analyses

The table contains six model specifications per biomarker (main analysis and S1–S5). Main-analysis diagnostics are retained in the `analysis_final/tables` snapshot; every sensitivity row is rebuilt from the matching diagnostics and metadata files under `analysis_final/RFBBP_result/prior_sensitivity`. No `analysis_simple` files are read.

In [ ]:
# Manuscript-ready MCMC diagnostic table: main analysis plus S1-S5.
# Inputs are restricted to analysis_final/RFBBP_result and analysis_final/tables.
import json

MAIN_DIAGNOSTIC_SNAPSHOT = TABLES / "rfbbp_main_and_prior_sensitivity_diagnostics.csv"


def diagnostic_table_row(indicator, analysis, diagnostics_path, metadata_path):
    diagnostics = read_diagnostic_series(diagnostics_path)
    with open(metadata_path, encoding="utf-8") as stream:
        metadata = json.load(stream)
    sampling = metadata["model_kwargs"]
    return {
        "Indicator": indicator,
        "analysis": analysis,
        "n": int(diagnostics["diag_n"]),
        "events": int(diagnostics["diag_n_events"]),
        "centers": int(diagnostics["diag_n_studies"]),
        "draws": int(sampling["draws"]),
        "tune": int(sampling["tune"]),
        "chains": int(sampling["chains"]),
        "target_accept": float(sampling["target_accept"]),
        "maximum Rhat": float(diagnostics["diag_max_r_hat"]),
        "minimum bulk ESS": int(diagnostics["diag_min_ess_bulk"]),
        "minimum tail ESS": int(diagnostics["diag_min_ess_tail"]),
        "beta_x Rhat": float(diagnostics["diag_beta_x_r_hat"]),
        "beta_x bulk ESS": int(diagnostics["diag_beta_x_ess_bulk"]),
        "divergences": int(diagnostics["diag_n_divergent"]),
        "minimum BFMI": float(diagnostics["diag_min_bfmi"]),
    }


# The three main-analysis diagnostic rows are retained in the analysis_final snapshot.
# Every S1-S5 row is rebuilt from its own diagnostics and metadata in RFBBP_result.
if not MAIN_DIAGNOSTIC_SNAPSHOT.exists():
    raise FileNotFoundError(f"Missing main-analysis diagnostic snapshot: {MAIN_DIAGNOSTIC_SNAPSHOT}")
main_snapshot = pd.read_csv(MAIN_DIAGNOSTIC_SNAPSHOT)
main_rows = main_snapshot.loc[main_snapshot["analysis"].eq("Main analysis")].copy()
if len(main_rows) != 3 or set(main_rows["Indicator"]) != set(main_analysis_variables):
    raise RuntimeError(
        "The main-analysis diagnostic snapshot must contain exactly one Main analysis row "
        "for Hemoglobin, Lactate, and Creatinine."
    )

diagnostic_rows = []
for indicator in main_analysis_variables:
    diagnostic_rows.append(main_rows.loc[main_rows["Indicator"].eq(indicator)].iloc[0].to_dict())
    for sensitivity_indicator, analysis, tag in prior_scenarios:
        if sensitivity_indicator != indicator:
            continue
        diagnostics_path = require_one(
            PRIOR_RESULT_DIR, f"*_{tag}_diagnostics.csv", f"diagnostics for {tag}"
        )
        metadata_path = require_one(
            PRIOR_RESULT_DIR, f"*_{tag}_metadata.json", f"metadata for {tag}"
        )
        diagnostic_rows.append(
            diagnostic_table_row(indicator, analysis, diagnostics_path, metadata_path)
        )

diagnostic_columns = [
    "Indicator", "analysis", "n", "events", "centers", "draws", "tune",
    "chains", "target_accept", "maximum Rhat", "minimum bulk ESS",
    "minimum tail ESS", "beta_x Rhat", "beta_x bulk ESS", "divergences",
    "minimum BFMI",
]
rfbbp_diagnostic_table = pd.DataFrame(diagnostic_rows, columns=diagnostic_columns)
if len(rfbbp_diagnostic_table) != 18:
    raise RuntimeError(f"Expected 18 diagnostic rows (3 biomarkers × 6 analyses); found {len(rfbbp_diagnostic_table)}")
if rfbbp_diagnostic_table.duplicated(["Indicator", "analysis"]).any():
    duplicates = rfbbp_diagnostic_table.loc[
        rfbbp_diagnostic_table.duplicated(["Indicator", "analysis"], keep=False),
        ["Indicator", "analysis"],
    ]
    raise RuntimeError(f"Duplicate diagnostic rows detected:\n{duplicates}")

for column in ["maximum Rhat", "beta_x Rhat", "minimum BFMI"]:
    rfbbp_diagnostic_table[column] = rfbbp_diagnostic_table[column].round(2)
rfbbp_diagnostic_table["target_accept"] = rfbbp_diagnostic_table["target_accept"].round(2)

diagnostic_csv = TABLES / "rfbbp_main_and_prior_sensitivity_diagnostics.csv"
diagnostic_xlsx = TABLES / "rfbbp_main_and_prior_sensitivity_diagnostics.xlsx"
rfbbp_diagnostic_table.to_csv(diagnostic_csv, index=False)
rfbbp_diagnostic_table.to_excel(diagnostic_xlsx, index=False)
display(rfbbp_diagnostic_table)


In [ ]:
rfbbp_rows = []
rfbbp_dir = RFBBP_RESULT
for variable, spec in BIOMARKERS.items():
    path = rfbbp_dir / f"{variable}_sofa_adjusted_mimic_iv_eicu_w_raw_rfbbp_hierarchical_5interval_no_center_filter_mimic_eicu_summary.csv"
    summary = pd.read_csv(path, index_col=0)
    estimate = summary.loc["HR_x"]
    rfbbp_rows.append({"variable": variable, "indicator": spec["label"], "method": "RFBBP",
                       "hr": estimate.estimate, "ci_low": estimate.CI_1, "ci_high": estimate.CI_2,
                       "log_hr": np.log(estimate.estimate)})
rfbbp_results = pd.DataFrame(rfbbp_rows)

four_methods = pd.concat([
    cox_results[["variable", "indicator", "method", "hr", "ci_low", "ci_high", "log_hr"]],
    meta_results.assign(method="Two-stage meta")[["variable", "indicator", "method", "hr", "ci_low", "ci_high", "coef"]]
        .rename(columns={"coef": "log_hr"}),
    rfbbp_results,
], ignore_index=True)
four_methods["effect_95ci"] = four_methods.apply(lambda r: f"{r.hr:.3f} ({r.ci_low:.3f}, {r.ci_high:.3f})", axis=1)
four_method_table = four_methods.pivot(index="indicator", columns="method", values="effect_95ci")
four_method_table = four_method_table.reindex(index=["Hemoglobin", "Lactate", "Creatinine"], columns=["Naive", "Naive_std", "Two-stage meta", "RFBBP"])
four_methods.to_csv(TABLES / "four_method_effects_three_biomarkers.csv", index=False)
four_method_table.to_csv(TABLES / "four_method_effects_article_table.csv")
display(four_method_table)

In [ ]:
def posterior_array(data_array, value_dim=None):
    stacked = data_array.stack(sample=("chain", "draw"))
    if value_dim is not None:
        stacked = stacked.transpose("sample", value_dim)
    return np.asarray(stacked.values)


def draw_summary(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    median, low, high = np.quantile(values, [0.50, 0.025, 0.975])
    return float(median), float(low), float(high)


attenuation_benchmark_rows = []
attenuation_rows = []

for biomarker_index, (variable, spec) in enumerate(BIOMARKERS.items()):
    # These centre parameters and posterior draws are RFBBP model outputs. All
    # traditional-model estimates and weights below are generated in this notebook.
    parameters = pd.read_csv(source_tables / f"rfbbp_absigma_center_parameters_{variable}_linear.csv")
    parameters = parameters.loc[
        parameters.database_group.eq("mimic+eicu")
        & parameters.center_filter_label.eq("no_filter")
    ].copy()
    posterior_path = Path(parameters.netcdf.iloc[0])
    if not posterior_path.exists():
        raise FileNotFoundError(f"Missing RFBBP posterior: {posterior_path}")

    point_parameters = parameters.pivot(index="idx", columns="parameter", values="estimate")
    point_parameters = point_parameters.dropna(subset=["b", "sigma_w"]).reset_index()
    center_ids = np.unique(cohorts[variable].center_id.astype(str).to_numpy())
    point_parameters["center_id"] = point_parameters["idx"].astype(int).map(
        lambda idx: center_ids[idx]
    )
    point_parameters["rho"] = point_parameters.b.pow(2) / (
        point_parameters.b.pow(2) + point_parameters.sigma_w.pow(2)
    )

    center_effects = two_stage_center_results_by_variable[variable].copy()
    center_effects["center_id"] = center_effects.center_id.astype(str)
    center_effects["information_weight"] = np.where(
        center_effects.status.eq("ok")
        & np.isfinite(center_effects.se)
        & center_effects.se.gt(0),
        1.0 / np.square(center_effects.se.astype(float)),
        np.nan,
    )
    weighted_centers = point_parameters.merge(
        center_effects[["center_id", "information_weight"]], on="center_id", how="left"
    )
    weighted_centers = weighted_centers.loc[
        weighted_centers.information_weight.notna()
        & weighted_centers.information_weight.gt(0)
    ].copy()
    center_index = weighted_centers.idx.astype(int).to_numpy()
    weights = weighted_centers.information_weight.astype(float).to_numpy()

    with xr.open_dataset(posterior_path, group="posterior") as posterior:
        b_dim = next(dim for dim in posterior["b"].dims if dim not in {"chain", "draw"})
        sigma_dim = next(dim for dim in posterior["sigma_w"].dims if dim not in {"chain", "draw"})
        b_draws = posterior_array(posterior["b"], b_dim)[:, center_index]
        sigma_draws = posterior_array(posterior["sigma_w"], sigma_dim)[:, center_index]
        beta_rfbbp_draws = posterior_array(posterior["beta_x"]).reshape(-1)

    rho_draws = np.square(b_draws) / (np.square(b_draws) + np.square(sigma_draws))
    attenuation_draws = np.sum(np.sqrt(rho_draws) * weights[None, :], axis=1) / weights.sum()
    predicted_draws = beta_rfbbp_draws * attenuation_draws

    naive = cox_results.loc[
        cox_results.variable.eq(variable) & cox_results.method.eq("Naive_std")
    ].iloc[0]
    rng = np.random.default_rng(20260901 + biomarker_index)
    naive_draws = rng.normal(float(naive.log_hr), float(naive.se), len(attenuation_draws))
    expected_latent_draws = naive_draws / attenuation_draws

    attenuation_factor, attenuation_low, attenuation_high = draw_summary(attenuation_draws)
    predicted_beta, predicted_low, predicted_high = draw_summary(predicted_draws)
    rfbbp_beta, rfbbp_low, rfbbp_high = draw_summary(beta_rfbbp_draws)
    expected_beta, expected_low, expected_high = draw_summary(expected_latent_draws)
    common = {
        "variable": variable,
        "indicator": spec["label"],
        "database_group": "mimic+eicu",
        "database_label": "MIMIC-IV + eICU",
        "rho_median": float(point_parameters.rho.median()),
        "attenuation_factor": attenuation_factor,
        "attenuation_ci_low": attenuation_low,
        "attenuation_ci_high": attenuation_high,
        "n_centers_weighted": int(len(weighted_centers)),
        "n": int(naive.n),
        "events": int(naive.events),
    }
    attenuation_benchmark_rows.extend([
        {**common, "comparison": "latent_scale", "method": "Expected corrected beta_x",
         "beta": expected_beta, "ci_low": expected_low, "ci_high": expected_high},
        {**common, "comparison": "latent_scale", "method": "Observed RFBBP beta_x",
         "beta": rfbbp_beta, "ci_low": rfbbp_low, "ci_high": rfbbp_high},
        {**common, "comparison": "observed_scale", "method": "Predicted observed beta",
         "beta": predicted_beta, "ci_low": predicted_low, "ci_high": predicted_high},
        {**common, "comparison": "observed_scale", "method": "Naive_std beta",
         "beta": float(naive.log_hr), "ci_low": float(np.log(naive.ci_low)),
         "ci_high": float(np.log(naive.ci_high))},
    ])
    attenuation_rows.extend([
        {"variable": variable, "indicator": spec["label"], "method": "RFBBP projection",
         "log_hr": predicted_beta, "ci_low": predicted_low, "ci_high": predicted_high,
         "rho_median": common["rho_median"], "attenuation_factor": attenuation_factor},
        {"variable": variable, "indicator": spec["label"], "method": "Naive_std Cox",
         "log_hr": float(naive.log_hr), "ci_low": float(np.log(naive.ci_low)),
         "ci_high": float(np.log(naive.ci_high)), "rho_median": common["rho_median"],
         "attenuation_factor": attenuation_factor},
    ])

attenuation_benchmark = pd.DataFrame(attenuation_benchmark_rows)
attenuation_benchmark.to_csv(TABLES / "attenuation_benchmarks_generated.csv", index=False)
attenuation = pd.DataFrame(attenuation_rows)
attenuation.to_csv(TABLES / "attenuation_naive_std_rfbbp.csv", index=False)

attenuation_x_limits = {
    "Hemoglobin": (-0.10, 0.00),
    "Lactate": (0.00, 0.70),
    "Creatinine": (0.00, 0.40),
}
attenuation_x_ticks = {
    "Hemoglobin": [-0.10, -0.075, -0.05, -0.025, 0.00],
    "Lactate": [0.00, 0.20, 0.40, 0.60],
    "Creatinine": [0.00, 0.10, 0.20, 0.30, 0.40],
}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, label in zip(axes, ["Hemoglobin", "Lactate", "Creatinine"]):
    panel = attenuation.loc[attenuation.indicator.eq(label)]
    for y, (method, color, marker) in zip([0.56, 0.44], [("RFBBP projection", COLORS["rfbbp"], "o"), ("Naive_std Cox", COLORS["naive"], "^")]):
        row = panel.loc[panel.method.eq(method)].iloc[0]
        ax.errorbar(row.log_hr, y, xerr=[[row.log_hr-row.ci_low], [row.ci_high-row.log_hr]], fmt=marker, color=color, capsize=0, ms=7)
        ax.text(row.log_hr, y+.045, f"{row.log_hr:.3f}", ha="center", va="bottom")
    factor = panel.attenuation_factor.iloc[0]
    ax.set_xlim(*attenuation_x_limits[label])
    ax.set_xticks(attenuation_x_ticks[label])
    ax.grid(False)
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", left=False)
    ax.set(yticks=[], ylim=(0.30, 0.70), xlabel="Log hazard ratio")
    ax.set_title(label, fontweight="bold", pad=25)
    ax.text(0.5, 1.035, f"attenuation factor={factor:.3f}", transform=ax.transAxes,
            ha="center", va="bottom", fontweight="normal")
legend_handles = [
    plt.Line2D([0], [0], color=COLORS["rfbbp"], marker="o", linestyle="-",
               label="RFBBP-predicted observed"),
    plt.Line2D([0], [0], color=COLORS["naive"], marker="^", linestyle="-",
               label="Naive_std Cox"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, .12, 1, 1)); fig.savefig(FIGURES / "attenuation_three_biomarkers.png", bbox_inches="tight"); plt.show()

## 8. Centre-specific equivalent thresholds and held-out consequences

In [ ]:
THRESHOLDS = {"Hemoglobin_hour_24": [8.0, 12.0], "Lactate_hour_24": [4.0, 8.0], "Creatinine_hour_24": [2.0, 2.5]}
MAIN_THRESHOLDS = {"Hemoglobin_hour_24": 8.0, "Lactate_hour_24": 4.0, "Creatinine_hour_24": 2.0}

threshold_rows = []
threshold_details = []
reclassification_details = []
validation_sets = {}
threshold_maps = {}
saved_split = pd.read_csv(source_tables / "three_biomarkers_train70_validate30_rfbbp_threshold_transportability_split_assignment.csv")
split_audit_rows = []
for variable, values in THRESHOLDS.items():
    slug = BIOMARKERS[variable]["label"].lower()
    params = pd.read_csv(source_tables / f"{slug}_train70_validate30_rfbbp_threshold_transportability_training_calibration_parameters.csv")
    cohort = cohorts[variable].copy()
    assignment = saved_split.loc[saved_split.biomarker.eq(BIOMARKERS[variable]["label"])].reset_index(drop=True)
    original = pd.read_csv(RFBBP_RESULT / f"{variable}_sofa_adjusted.csv")
    original_ordered = pd.concat([group for _, group in original.groupby("center_id", sort=False)], ignore_index=True)
    same_rows = (len(original_ordered) == len(assignment) and
                 original_ordered.database.astype(str).equals(assignment.database.astype(str)) and
                 original_ordered.center_id.astype(str).equals(assignment.center_id.astype(str)) and
                 np.array_equal(original_ordered.event.to_numpy(), assignment.event.to_numpy()) and
                 np.allclose(original_ordered.time, assignment.time, rtol=0, atol=1e-10) and
                 np.allclose(original_ordered.exposure_analysis, assignment.exposure_analysis, rtol=0, atol=1e-10))
    if not same_rows:
        raise ValueError(f"Saved split rows do not align with the original {variable} analysis dataset")
    split_by_stay = original_ordered[["database", "stay_id"]].copy()
    split_by_stay["split"] = assignment.split.to_numpy()
    cohort = cohort.merge(split_by_stay, on=["database", "stay_id"], how="left", validate="one_to_one")
    matched_n = int(cohort.split.notna().sum())
    split_audit_rows.append({"variable": variable, "cohort_n": len(cohort), "assignment_n": len(assignment),
                             "saved_rows_aligned": same_rows, "matched_n": matched_n,
                             "validation_n": int(cohort.split.eq("validation").sum())})
    if matched_n != len(cohort) or len(assignment) != len(cohort):
        raise ValueError(f"Saved split does not match the {variable} analysis cohort one-to-one")
    validation = cohort.loc[cohort.split.eq("validation")].merge(params[["center_id", "a", "b", "sigma_w"]], on="center_id", how="inner")
    validation_sets[variable] = validation

    for reference_value in values:
        reference_analysis = np.log(reference_value) if BIOMARKERS[variable]["scale"] == "log" else reference_value
        latent_states = (reference_analysis - params.a) / params.b
        common_state = float(latent_states.median())
        centre_threshold_analysis = params.a + params.b * common_state
        centre_threshold_raw = np.exp(centre_threshold_analysis) if BIOMARKERS[variable]["scale"] == "log" else centre_threshold_analysis
        centre_thresholds = params[["center_id", "database"]].copy()
        centre_thresholds["equivalent_threshold"] = centre_threshold_raw
        centre_thresholds["reference_value"] = reference_value
        centre_thresholds["variable"] = variable
        threshold_details.append(centre_thresholds)
        threshold_maps[(variable, reference_value)] = centre_thresholds.set_index("center_id")["equivalent_threshold"]

        assessed = validation.merge(centre_thresholds[["center_id", "equivalent_threshold"]], on="center_id", how="inner")
        low_is_abnormal = variable == "Hemoglobin_hour_24"
        uniform_flag = assessed.exposure_raw.le(reference_value) if low_is_abnormal else assessed.exposure_raw.ge(reference_value)
        calibrated_flag = assessed.exposure_raw.le(assessed.equivalent_threshold) if low_is_abnormal else assessed.exposure_raw.ge(assessed.equivalent_threshold)
        assessed["reclassified"] = uniform_flag.ne(calibrated_flag)
        by_centre = assessed.groupby(["database", "center_id"]).reclassified.agg(["size", "sum", "mean"]).reset_index()
        by_centre["variable"] = variable
        by_centre["reference_value"] = reference_value
        by_centre["reclassified_percent"] = 100 * by_centre["mean"]
        reclassification_details.append(by_centre)
        threshold_rows.append({"variable": variable, "biomarker": BIOMARKERS[variable]["label"],
                               "reference_value": reference_value, "unit": BIOMARKERS[variable]["unit"],
                               "equivalent_min": centre_threshold_raw.min(), "equivalent_p25": centre_threshold_raw.quantile(.25),
                               "equivalent_median": centre_threshold_raw.median(), "equivalent_p75": centre_threshold_raw.quantile(.75),
                               "equivalent_max": centre_threshold_raw.max(), "validation_n": len(assessed),
                               "reclassified_n": int(assessed.reclassified.sum()),
                               "reclassified_percent": 100 * assessed.reclassified.mean()})

threshold_summary = pd.DataFrame(threshold_rows)
centre_equivalent_thresholds = pd.concat(threshold_details, ignore_index=True)
reclassification_by_centre = pd.concat(reclassification_details, ignore_index=True)
split_audit = pd.DataFrame(split_audit_rows)
threshold_summary.to_csv(TABLES / "requested_threshold_transportability_summary.csv", index=False)
centre_equivalent_thresholds.to_csv(TABLES / "requested_centre_equivalent_thresholds.csv", index=False)
reclassification_by_centre.to_csv(TABLES / "requested_reclassification_by_centre.csv", index=False)
split_audit.to_csv(TABLES / "saved_validation_split_match_audit.csv", index=False)
display(split_audit, threshold_summary)

In [ ]:
profile_rows = []
profile_center_rows = []
profile_grids = {"Hemoglobin_hour_24": np.arange(7, 13.1, .25),
                 "Lactate_hour_24": np.arange(2, 8.1, .25),
                 "Creatinine_hour_24": np.arange(1, 3.05, .1)}
for variable, grid in profile_grids.items():
    slug = BIOMARKERS[variable]["label"].lower()
    params = pd.read_csv(source_tables / f"{slug}_train70_validate30_rfbbp_threshold_transportability_training_calibration_parameters.csv")
    for reference_value in grid:
        reference_analysis = np.log(reference_value) if BIOMARKERS[variable]["scale"] == "log" else reference_value
        common_state = ((reference_analysis - params.a) / params.b).median()
        equivalent = params.a + params.b * common_state
        if BIOMARKERS[variable]["scale"] == "log":
            equivalent = np.exp(equivalent)
        for center_id, database, value in zip(params.center_id, params.database, equivalent):
            profile_center_rows.append({"variable": variable, "biomarker": BIOMARKERS[variable]["label"],
                                        "center_id": center_id, "database": database,
                                        "reference_value": reference_value, "equivalent_value": value})
        profile_rows.append({"variable": variable, "biomarker": BIOMARKERS[variable]["label"],
                             "reference_value": reference_value, "equivalent_p10": equivalent.quantile(.10),
                             "equivalent_p25": equivalent.quantile(.25), "equivalent_median": equivalent.median(),
                             "equivalent_p75": equivalent.quantile(.75), "equivalent_p90": equivalent.quantile(.90)})
threshold_profiles = pd.DataFrame(profile_rows)
threshold_profile_centres = pd.DataFrame(profile_center_rows)
threshold_profiles.to_csv(TABLES / "three_biomarker_threshold_profiles.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
highlight_values = {"Lactate": 4.0, "Hemoglobin": 10.0, "Creatinine": 1.0}
for panel, (ax, label) in enumerate(zip(axes, ["Lactate", "Hemoglobin", "Creatinine"]), start=1):
    p = threshold_profiles.loc[threshold_profiles.biomarker.eq(label)].sort_values("reference_value")
    centre_profiles = threshold_profile_centres.loc[(threshold_profile_centres.biomarker.eq(label)) & threshold_profile_centres.database.eq("eicu")]
    unit = BIOMARKERS[[k for k,v in BIOMARKERS.items() if v['label']==label][0]]["unit"]
    for _, centre in centre_profiles.groupby("center_id", sort=False):
        ax.plot(centre.reference_value, centre.equivalent_value, color="#9AA3AD", lw=.25, alpha=.028, zorder=1)
    ax.plot(p.reference_value, p.reference_value, color="#555555", lw=.85, ls=(0, (4, 4)), alpha=.42,
            label="_nolegend_", zorder=2)
    ax.fill_between(p.reference_value, p.equivalent_p10, p.equivalent_p90, color="#E8EDF2", alpha=.48,
                    label="10th–90th percentile", zorder=2)
    ax.fill_between(p.reference_value, p.equivalent_p25, p.equivalent_p75, color="#B8D2E8", alpha=.52,
                    label="Interquartile range", zorder=3)
    ax.plot(p.reference_value, p.equivalent_median, color="#4D8FC5", lw=1.45, zorder=4)
    highlight = highlight_values[label]
    nearest = p.iloc[(p.reference_value-highlight).abs().argmin()]
    ax.axvspan(highlight-(p.reference_value.diff().median()/2), highlight+(p.reference_value.diff().median()/2), color="#f1c75b", alpha=.18)
    ax.annotate(f"{highlight:g} {unit} reference\nCentral 80%: {nearest.equivalent_p10:.2f}–{nearest.equivalent_p90:.2f} {unit}",
                xy=(highlight, nearest.equivalent_p10), xytext=(highlight + .08*(p.reference_value.max()-p.reference_value.min()), nearest.equivalent_p10),
                fontsize=7.5, arrowprops={"arrowstyle":"-", "color":"#6b4ea0", "lw":.7})
    ax.set(title=f"{chr(64+panel)}. {label}", xlabel=f"Common clinical threshold ({unit})",
           ylabel="Centre-specific equivalent threshold")
    band_low, band_high = p.equivalent_p10.min(), p.equivalent_p90.max()
    band_pad = .08 * (band_high - band_low)
    ax.set_ylim(band_low-band_pad, band_high+band_pad)
    inset_x = .16 if label == "Creatinine" else .07
    inset = ax.inset_axes([inset_x, .70, .27, .20], facecolor="white")
    width = p.equivalent_p90 - p.equivalent_p10
    inset.plot(p.reference_value, width, color="#6b4ea0", marker="o", markevery=4, ms=2.2, lw=1)
    inset.set_title(f"Central 80% span ({unit})", fontsize=6.2, loc="left", fontweight="bold")
    y_pad = {"Lactate": .20, "Creatinine": .12, "Hemoglobin": .12}[label]
    inset.set_ylim(0, width.max()+y_pad)
    inset.tick_params(labelsize=5.5); inset.grid(True, lw=.4)
    inset.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False)
fig.subplots_adjust(left=.075, right=.985, top=.90, bottom=.15, wspace=.24)
fig.savefig(FIGURES / "three_biomarker_threshold_transportability_profiles.png", bbox_inches="tight"); plt.show()

In [ ]:
threshold_effect_rows = []
for variable, reference_values in THRESHOLDS.items():
    for reference_value in reference_values:
        validation = validation_sets[variable].copy()
        validation["equivalent_threshold"] = validation.center_id.map(threshold_maps[(variable, reference_value)])
        low_is_abnormal = variable == "Hemoglobin_hour_24"
        validation["uniform_flag"] = validation.exposure_raw.le(reference_value) if low_is_abnormal else validation.exposure_raw.ge(reference_value)
        validation["equivalent_flag"] = validation.exposure_raw.le(validation.equivalent_threshold) if low_is_abnormal else validation.exposure_raw.ge(validation.equivalent_threshold)
        for center_id, centre in validation.groupby("center_id"):
            for method, exposure_col in [("Uniform", "uniform_flag"), ("RFBBP-equivalent", "equivalent_flag")]:
                high_risk = centre[exposure_col].astype(bool)
                n, events = len(centre), int(centre.event.sum())
                high_n, reference_n = int(high_risk.sum()), int((~high_risk).sum())
                high_events = int(centre.loc[high_risk, "event"].sum())
                reference_events = int(centre.loc[~high_risk, "event"].sum())
                row = {"variable": variable, "reference_value": reference_value,
                       "indicator": BIOMARKERS[variable]["label"], "center_id": center_id,
                       "method": method, "n": n, "events": events, "nonevents": n-events,
                       "high_risk_n": high_n, "reference_n": reference_n,
                       "high_risk_events": high_events, "reference_events": reference_events}
                stable = (n >= 20 and events >= 5 and n-events >= 5 and high_n >= 5 and reference_n >= 5 and
                          high_events >= 1 and reference_events >= 1)
                if not stable:
                    row["status"] = "skipped_sparse_threshold_groups"
                else:
                    terms, info = _cox(centre.assign(**{exposure_col: high_risk.astype(float)}), exposure_col,
                                       covariates_for(variable, "sofa_adjusted"), strata_center=False)
                    row["status"] = info["status"]
                    if info["status"] == "ok":
                        estimate = terms.loc[terms.term.eq(exposure_col)].iloc[0]
                        finite = np.isfinite(estimate.coef) and np.isfinite(estimate.se) and estimate.se > 0
                        row["status"] = "ok" if finite else "failed_nonfinite"
                        if finite:
                            row.update({"log_hr": estimate.coef, "se": estimate.se, "hr": estimate.hr})
                threshold_effect_rows.append(row)

threshold_effects = pd.DataFrame(threshold_effect_rows)
threshold_effects["common_center"] = False

threshold_meta_rows = []
for variable, reference_values in THRESHOLDS.items():
  for reference_value in reference_values:
    label, unit = BIOMARKERS[variable]["label"], BIOMARKERS[variable]["unit"]
    thresholds = centre_equivalent_thresholds.loc[(centre_equivalent_thresholds.variable.eq(variable)) &
                                                   centre_equivalent_thresholds.reference_value.eq(reference_value)].sort_values("equivalent_threshold")
    reclass = reclassification_by_centre.loc[(reclassification_by_centre.variable.eq(variable)) &
                                              reclassification_by_centre.reference_value.eq(reference_value)].sort_values("reclassified_percent")
    all_effects = threshold_effects.loc[(threshold_effects.variable.eq(variable)) &
                                        threshold_effects.reference_value.eq(reference_value)]
    effects = all_effects.loc[all_effects.status.eq("ok") & np.isfinite(all_effects.log_hr) &
                              np.isfinite(all_effects.se) & all_effects.se.gt(0)].copy()
    # Treat implausibly large/small HRs as numerical separation rather than
    # estimable effects.  Pairwise comparison requires both methods to pass.
    effects["hr_in_plot_range"] = effects.hr.between(0.01, 100, inclusive="both")
    uniform_centres = set(effects.loc[effects.method.eq("Uniform"), "center_id"])
    rfbbp_centres = set(effects.loc[effects.method.eq("RFBBP-equivalent"), "center_id"])
    range_ok = (effects.pivot(index="center_id", columns="method", values="hr_in_plot_range")
                .reindex(columns=["Uniform", "RFBBP-equivalent"]).fillna(False).all(axis=1))
    common_centres = sorted((uniform_centres & rfbbp_centres) & set(range_ok.index[range_ok]))
    effects["common_center"] = effects.center_id.isin(common_centres)
    threshold_effects.loc[effects.index, "common_center"] = effects["common_center"]
    paired = (effects.loc[effects.common_center]
              .pivot(index="center_id", columns="method", values="log_hr")
              .loc[common_centres])
    paired_se = (effects.loc[effects.common_center]
                 .pivot(index="center_id", columns="method", values="se")
                 .loc[common_centres])

    meta_labels = {}
    meta_stats = {}
    for method in ["Uniform", "RFBBP-equivalent"]:
        meta_data = effects.loc[effects.method.eq(method) & effects.common_center, ["log_hr", "se"]]
        yi, vi = meta_data.log_hr.to_numpy(), meta_data.se.to_numpy() ** 2
        weights = 1 / vi
        fixed = np.sum(weights * yi) / np.sum(weights)
        q = np.sum(weights * (yi-fixed)**2)
        c = np.sum(weights) - np.sum(weights**2) / np.sum(weights)
        tau2 = max(0, (q-(len(yi)-1))/c) if len(yi) > 1 and c > 0 else 0
        i2 = max(0, (q-(len(yi)-1))/q)*100 if q > 0 and len(yi) > 1 else 0
        meta_labels[method] = f"$\\tau^2$={tau2:.3f}; $I^2$={i2:.1f}%"
        meta_stats[method] = {"tau2": tau2, "I2": i2}
        threshold_meta_rows.append({"variable": variable, "reference_value": reference_value,
                                    "method": method, "common_centres": len(common_centres),
                                    "centres": len(yi), "tau2": tau2, "I2": i2})

    fig = plt.figure(figsize=(12.8, 8.0), dpi=300)
    gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.18], hspace=.48, wspace=.28)
    axes = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[1, :])]
    eq_p10 = thresholds.equivalent_threshold.quantile(.10)
    eq_p90 = thresholds.equivalent_threshold.quantile(.90)
    # The clinical reference is a single value (dashed line).  Shaded regions
    # describe the cross-centre distribution of RFBBP-equivalent local values.
    axes[0].axvspan(eq_p10, eq_p90, color="#DDE3E9", alpha=.72,
                    label=f"Central 80% of centres: {eq_p10:.2f}–{eq_p90:.2f} {unit}", zorder=0)
    axes[0].scatter(thresholds.equivalent_threshold, np.arange(len(thresholds)), s=14,
                    color="#4f91c7", alpha=.90,
                    label=f"Range: {thresholds.equivalent_threshold.min():.2f}–{thresholds.equivalent_threshold.max():.2f} {unit}",
                    zorder=2)
    axes[0].axvline(reference_value, color="#444444", ls="--", lw=1.4,
                    label=f"Fixed reference: {reference_value:g} {unit}", zorder=3)
    axes[0].set(title="A. Training-centre equivalent values",
                xlabel="Equivalent local threshold", ylabel="Centres ranked")
    axes[0].yaxis.grid(False)
    axes[0].legend(frameon=False, loc="upper left", fontsize=7.5)
    nonzero_reclass = reclass.loc[reclass.reclassified_percent.gt(0)].copy()
    axes[1].scatter(nonzero_reclass.reclassified_percent, np.arange(len(nonzero_reclass)),
                    s=18, color="#4f91c7")
    overall = threshold_summary.loc[(threshold_summary.variable.eq(variable)) & threshold_summary.reference_value.eq(reference_value), "reclassified_percent"].iloc[0]
    axes[1].axvline(overall, color="#6b4ea0", lw=1.5)
    axes[1].set(title="B. Held-out reclassification",
                xlabel="Validation patients reclassified (%)", ylabel="Centres with >0% reclassification")
    axes[1].yaxis.grid(False)
    axes[1].text(overall + .15, max(len(nonzero_reclass) * .05, .5), f"Patient-weighted: {overall:.1f}%",
                 rotation=90, va="bottom", ha="left", color="#6b4ea0", fontsize=8)
    order = paired.sort_values("Uniform").index
    paired = paired.loc[order].reset_index(drop=True)
    paired_se = paired_se.loc[order].reset_index(drop=True)
    x = np.arange(len(paired))
    axes[2].errorbar(x-.10, paired["Uniform"], yerr=1.96*paired_se["Uniform"], fmt="o",
                     color="#9aa6b2", ecolor="#c9d0d7", alpha=.72, ms=3.4, lw=.40, capsize=0,
                     label=f"Uniform {reference_value:g} {unit} ({meta_labels['Uniform']})")
    axes[2].errorbar(x+.10, paired["RFBBP-equivalent"], yerr=1.96*paired_se["RFBBP-equivalent"], fmt="o",
                     color="#6b4ea0", ecolor="#b9a8d7", alpha=.82, ms=3.4, lw=.40, capsize=0,
                     label=f"RFBBP-equivalent threshold ({meta_labels['RFBBP-equivalent']})")
    for i, row in paired.iterrows():
        axes[2].plot([i, i], [row["Uniform"], row["RFBBP-equivalent"]],
                     color="#7f8994", lw=.85, alpha=.72, zorder=1)
    axes[2].axhline(0, color="#555555", ls="--")
    point_low = float(paired[["Uniform", "RFBBP-equivalent"]].min().min())
    point_high = float(paired[["Uniform", "RFBBP-equivalent"]].max().max())
    point_pad = max(.35, .10 * (point_high - point_low))
    axes[2].set_ylim(point_low - point_pad, point_high + point_pad)
    tau_uniform = meta_stats["Uniform"]["tau2"]
    tau_rfbbp = meta_stats["RFBBP-equivalent"]["tau2"]
    tau_reduction = 100 * (tau_uniform - tau_rfbbp) / tau_uniform if tau_uniform > 0 else 0
    i2_change = meta_stats["RFBBP-equivalent"]["I2"] - meta_stats["Uniform"]["I2"]
    axes[2].set(xlabel="Paired estimable centres ordered by uniform-threshold estimate",
                ylabel="Adjusted log HR, high risk vs reference",
                title="C. Paired held-out prognostic contrasts")
    axes[2].text(.985, .06,
                 "Confidence intervals extending beyond the plotting range were clipped for display.",
                 transform=axes[2].transAxes, ha="right", va="bottom", fontsize=8.5,
                 bbox={"facecolor": "white", "alpha": .90, "edgecolor": "none"})
    axes[2].legend(frameon=False, loc="upper left")
    fig.suptitle(label, fontsize=15, fontweight="bold", y=.995)
    fig.subplots_adjust(left=.08, right=.98, top=.93, bottom=.08)
    suffix = str(reference_value).replace(".", "p")
    output_name = f"{label.lower()}_threshold_{suffix}_transportability.png"
    fig.savefig(FIGURES / output_name, bbox_inches="tight")
    if reference_value == MAIN_THRESHOLDS[variable]:
        fig.savefig(FIGURES / f"{label.lower()}_threshold_transportability_main.png", bbox_inches="tight")
    plt.show()
threshold_effects.to_csv(TABLES / "heldout_main_threshold_centre_effects.csv", index=False)
pd.DataFrame(threshold_meta_rows).to_csv(TABLES / "heldout_threshold_meta_heterogeneity.csv", index=False)